In [13]:
import sys
import os
import slurm_utils

HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
print(eid)
filenames = fit_eid(eid,sessdf)

print('saving to files:',filenames)
"""

ADD_TO_SAVING_PATH = 'eid20_AdjustFeatures'
SUBDIRECTORY = dut.decoding_details(TARGET,MODEL,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,TIME_WINDOW,
                             ADD_TO_SAVING_PATH)
SLURM_DIRECTORY = os.path.join(SCRATCH,'international-brain-lab/prior-localization/slurm/')
print(SUBDIRECTORY)
SUBDIRECTORY=''

decode_pLeft_task_Logistic_control_signcont_1_pseudosessions_align_goCue_times_timeWindow_-0_6_-0_2_eid20_AdjustFeatures


In [14]:
# make python file
py_file = os.path.join(SLURM_DIRECTORY, SUBDIRECTORY, '_'.join(['slurm_decode_eid',''])+'.py')
if not os.path.exists(os.path.join(SLURM_DIRECTORY, SUBDIRECTORY)):
    os.mkdir(os.path.join(SLURM_DIRECTORY, SUBDIRECTORY))
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)

sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')
#
# sessdf_eids=np.unique(sessdf_eids)[30:32]
sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
            '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
            '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
            'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
            '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
            '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
            'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
            'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
            '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
            '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']

# submit python file with eid inputs
for eid in sessdf_eids:
    slurm_utils.slurm_submit_python_file(py_file, 
                             eid, 
                             ADD_TO_SAVING_PATH = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 8,
                             SLURM_DIRECTORY = SLURM_DIRECTORY,
                             SUBDIRECTORY = SUBDIRECTORY)
    

Submitted batch job 45115495
Submitted batch job 45115497
Submitted batch job 45115498
Submitted batch job 45115499
Submitted batch job 45115500
Submitted batch job 45115501
Submitted batch job 45115502
Submitted batch job 45115503
Submitted batch job 45115504
Submitted batch job 45115505
Submitted batch job 45115506
Submitted batch job 45115507
Submitted batch job 45115509
Submitted batch job 45115510
Submitted batch job 45115511
Submitted batch job 45115512
Submitted batch job 45115514
Submitted batch job 45115516
Submitted batch job 45115517
Submitted batch job 45115518


In [21]:

os.system("squeue -u $USER");
slurm_utils.get_decoding_output_files(SLURM_DIRECTORY = SLURM_DIRECTORY,
                                      SUBDIRECTORY = SUBDIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          45110503    normal interact  bensonb  R    1:29:59      1 sh02-01n56
          45098716    normal interact  bensonb  R    3:23:15      1 sh02-01n50


[]

In [12]:

finished = get_decoding_output_files(label = '',
                                     SLURM_DIRECTORY = SLURM_DIRECTORY,
                                     SUBDIRECTORY = SUBDIRECTORY)
#%%
indexers = ['subject', 'eid', 'probe', 'region']
indexers_neurometric = ['low_slope', 'high_slope', 'low_range', 'high_range', 'shift']
resultslist = []
for fn in finished:
    fo = open(fn, 'rb')
    result = pickle.load(fo)
    fo.close()
    tmpdict = {**{x: result[x] for x in indexers},
               'fold': -1,
               'mask': ''.join([str(item) for item in list(result['fit']['mask'].values * 1)]),
               'Rsquared_test': result['fit']['Rsquared_test_full'],
               **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquared_test_full']
                  for i in range(N_PSEUDO)}}
    if result['fit']['full_neurometric'] is not None \
            and np.all([result['pseudosessions'][i]['full_neurometric'] is not None for i in range(N_PSEUDO)]):
        tmpdict = {**tmpdict,
                   **{idx_neuro: result['fit']['full_neurometric'][idx_neuro]
                      for idx_neuro in indexers_neurometric},
                   **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i]['full_neurometric'][idx_neuro]
                      for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}}
    resultslist.append(tmpdict)
    for kfold in range(result['fit']['nFolds']):
        tmpdict = {**{x: result[x] for x in indexers},
                   'fold': kfold,
                   'Rsquared_test': result['fit']['Rsquareds_test'][kfold],
                   'Best_regulCoef': result['fit']['best_params'][kfold],
                   **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquareds_test'][kfold]
                      for i in range(N_PSEUDO)},
                   }
        if result['fit']['fold_neurometric'] is not None:
            tmpdict = {**tmpdict,
                       **{idx_neuro: result['fit']['fold_neurometric'][kfold][idx_neuro]
                          for idx_neuro in indexers_neurometric}}
        if np.all([result['pseudosessions'][i]['fold_neurometric'] is not None for i in range(N_PSEUDO)]):
            tmpdict = {**tmpdict,
                       **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i][
                           'fold_neurometric'][kfold][idx_neuro]
                          for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}
                       }
        resultslist.append(tmpdict)
resultsdf = pd.DataFrame(resultslist).set_index(indexers)

start_tw, end_tw = TIME_WINDOW
fn = OUTPUT_PATH + '_'.join([DATE, dut.decoding_details(TARGET,MODEL,
                                                        ESTIMATORSTR,
                                                        ALIGN_TIME,
                                                        CONTROL_FEATURES,
                                                        N_PSEUDO,TIME_WINDOW,
                                                        ADD_TO_SAVING_PATH)])
fn = fn + '.parquet'

metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
resultsdf.to_parquet(fn)
metadata_df.to_pickle(metadata_fn)

